# Akan (Asante Twi) TTS — Fine-tuning facebook/mms-tts-aka

**Author:** Abdul Rashid Dickson · **Task:** UG MSc stage-2 TTS assignment

Pipeline: prepared single-narrator corpus (~5–10 h Asante Twi, GhanaNLP *ghana-speech*, CC-BY-NC-4.0)
→ fine-tune Meta's `mms-tts-aka` VITS checkpoint with the `ylacombe/finetune-hf-vits` harness.

Runtime: **GPU T4** (Runtime → Change runtime type → T4 GPU). Checkpoints save to Google Drive,
so the run survives session limits and can be resumed.

In [ ]:
# ═══════════════ RUN-EVERYTHING — THE ONLY CELL YOU RUN ═══════════════
# Press play on THIS cell. Every time. Any number of times. It knows the order:
# fresh session → runs everything; interrupted → continues; mid-training → just monitors.
# (The cells below are kept as reference - do not run them.)
import os, sys, json, glob, subprocess, time, shutil, urllib.request
from pathlib import Path

def stage(n, t): print("\n" + "="*14 + f"  STAGE {n}: {t}  " + "="*14)

def shell(cmd, must=True, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0 and must:
        print("COMMAND FAILED:", cmd); print(r.stdout[-1500:] if r.stdout else ""); print(r.stderr[-1500:])
        raise SystemExit("Send the lines above to Hermes in Telegram.")
    return r

# ── S1: GPU ──────────────────────────────────────────────────────────
stage(1, "GPU check")
import torch
if not torch.cuda.is_available():
    raise SystemExit("NO GPU — Runtime menu → 'Change runtime type' → T4 GPU → save → run this cell again.")
print("GPU OK:", torch.cuda.get_device_name(0))

# ── S2: harness + pin (skips when already done) ──────────────────────
stage(2, "Harness + versions")
import importlib.metadata as md
need_install = True
try:
    if md.version("transformers").startswith("4.49"):
        need_install = Path("/content/finetune-hf-vits/run_vits_finetuning.py").exists()
        if not need_install: print("transformers 4.49 ok; harness missing → cloning only")
        else: print("transformers 4.49 + harness present → skip"); 
except Exception:
    print("transformers wrong/missing → full install")
if need_install:
    if not Path("/content/finetune-hf-vits/run_vits_finetuning.py").exists():
        shell("git clone --depth 1 https://github.com/ylacombe/finetune-hf-vits /content/finetune-hf-vits")
        shell("pip install -q -r /content/finetune-hf-vits/requirements.txt")
        shell("pip install -q 'datasets[audio]>=3.2,<4.0' librosa soundfile accelerate", must=False)
    shell("pip install -q 'transformers==4.49.0'")
    shell("pip install -q 'datasets[audio]>=3.2,<4.0' librosa soundfile accelerate", must=False)
if not glob.glob("/content/finetune-hf-vits/monotonic_align/monotonic_align/core*.so"):
    shell("mkdir -p monotonic_align && python setup.py build_ext --inplace", cwd="/content/finetune-hf-vits/monotonic_align")
shell("mkdir -p /content/finetune-hf-vits/monotonic_align/monotonic_align")
shell("cp -n /content/finetune-hf-vits/monotonic_align/monotonic_align/core.py /content/finetune-hf-vits/monotonic_align/monotonic_align/core.py 2>/dev/null || true")
import transformers, datasets
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__)
assert transformers.__version__.startswith("4."), "wrong transformers — rerun after pin"

# ── S3: HF login (skips if token valid) ──────────────────────────────
stage(3, "Hugging Face login")
from huggingface_hub import HfApi
api = HfApi()
try:
    me = api.whoami()
    print("already logged in as:", me["name"])
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()
    api = HfApi()
    try:
        me = api.whoami(); print("logged in as:", me["name"])
    except Exception:
        raise SystemExit("Login did not complete — run this cell again and finish the login.")

# ── S4: corpus (skips if built) ──────────────────────────────────────
stage(4, "Corpus (mount Drive → unzip → stream audio)")
if not Path("/content/tts_data").exists():
    from google.colab import drive
    drive.mount("/content/drive")
    shell("mkdir -p /content/work && unzip -o -q '/content/drive/MyDrive/UG_TTS/tts_data.zip' -d /content/work")
    base = "https://raw.githubusercontent.com/Dickson32-cell/akan-tts-asante/master/src/"
    for f in ["normalize_twi.py", "extract_asante_colab.py"]:
        Path("/content/" + f).write_text(urllib.request.urlopen(base + f, timeout=60).read().decode("utf-8"), encoding="utf-8")
    r = shell("python /content/extract_asante_colab.py --manifest /content/work/selected.jsonl --out /content/tts_data", must=False)
    if r.returncode != 0:
        print(r.stdout[-800:] if r.stdout else "", r.stderr[-800:] if r.stderr else "")
        raise SystemExit("corpus build failed — send lines above to Hermes")
else:
    print("tts_data present → skip")

# ── S5: base training checkpoint (skips if built) ────────────────────
stage(5, "Training base (generator + Meta discriminator for 'aka')")
if not Path("/content/aka_train_base").exists():
    shell("python /content/finetune-hf-vits/convert_original_discriminator_checkpoint.py --language_code aka --pytorch_dump_folder_path /content/aka_train_base",
          cwd="/content/finetune-hf-vits")
    print("base checkpoint built:", os.listdir("/content/aka_train_base"))
else:
    print("base checkpoint present → skip")

# ── S6: config (always rewritten — gentle settings locked) ───────────
stage(6, "Config (hub id fixed; gentle LR run)")
cfg = {
  "project_name": "akan_twi_tts", "push_to_hub": True,
  "hub_model_id": "Dickson32-cell/akan-twi-mms",
  "report_to": ["tensorboard"], "logging_strategy": "steps", "logging_steps": 20,
  "overwrite_output_dir": False,
  "output_dir": "/content/akan-vits-finetuned",
  "dataset_name": "/content/tts_data", "dataset_config_name": None,
  "audio_column_name": "audio", "text_column_name": "text",
  "train_split_name": "train", "eval_split_name": "eval",
  "speaker_id_column_name": "speaker_id", "filter_on_speaker_id": 0,
  "override_speaker_embeddings": False,
  "full_generation_sample_text": "Mema wo akye, me nuanom. Wo ho te sɛn?",
  "max_duration_in_seconds": 12.0, "min_duration_in_seconds": 2.0, "max_tokens_length": 500,
  "model_name_or_path": "/content/aka_train_base",
  "preprocessing_num_workers": 2,
  "do_train": True, "num_train_epochs": 20,
  "gradient_accumulation_steps": 1, "per_device_train_batch_size": 16,
  "learning_rate": 1e-4, "lr_scheduler_type": "cosine", "warmup_ratio": 0.02,
  "adam_beta1": 0.8, "adam_beta2": 0.99, "weight_decay": 0.01, "group_by_length": False,
  "do_eval": True, "eval_steps": 500, "per_device_eval_batch_size": 8,
  "max_eval_samples": 25, "do_step_schedule_per_epoch": False,
  "save_steps": 1000, "save_total_limit": 3,
  "weight_disc": 3, "weight_fmaps": 1, "weight_gen": 1, "weight_kl": 1.5,
  "weight_duration": 1, "weight_mel": 35, "fp16": True, "seed": 456
}
Path("/content/finetune_akan.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
print("config written (LR 1e-4, 20 epochs, repo Dickson32-cell/akan-twi-mms)")

# ── S7: training — launch fresh / resume / or just monitor ───────────
stage(7, "Training")
LOG = "/content/train_akan.log"
pid_file = "/content/train_pid.txt"
RUNV = "run_vits_finetuning.py"
already = False
if Path(pid_file).exists():
    try:
        old = int(Path(pid_file).read_text().strip())
        c = Path(f"/proc/{old}/cmdline")
        if c.exists() and RUNV in c.read_text(errors="replace").replace("\x00", " "):
            proc = None; already = True
            print("training ALREADY RUNNING (pid", old, ") — switching to monitor mode")
        else:
            print("stale pid file → fresh launch")
    except Exception:
        pass

def monitor(proc, pid=None):
    if pid is not None: 
        pass
    last = 0
    while True:
        if proc is not None and proc.poll() is not None: break
        time.sleep(25)
        try:
            lines = Path(LOG).read_text(errors="replace").splitlines()
        except FileNotFoundError:
            continue
        for l in lines[last:]:
            if any(k in l.lower() for k in ("loss", "eval_", "error", "traceback", "epoch", "it/s")):
                print(l[:180])
        last = len(lines)
        if proc is None and not Path("/proc/%d/cmdline" % old).exists():
            break
    return

if already:
    monitor(None, None)
    print("training process ENDED — continuing to convergence report")
else:
    cmd = ["python", "-m", "accelerate.commands.launch", "/content/finetune-hf-vits/run_vits_finetuning.py",
           "/content/finetune_akan.json", "--save_steps", "1000", "--save_total_limit", "3", "--logging_steps", "20"]
    if glob.glob("/content/akan-vits-finetuned/checkpoint-*"):
        cmd += ["--resume_from_checkpoint", "true"]
        print("checkpoints found → RESUME")
    else:
        print("no checkpoints → FRESH RUN")
    log_f = open(LOG, "a")
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, bufsize=1,
                            universal_newlines=True, cwd="/content/finetune-hf-vits")
    Path(pid_file).write_text(str(proc.pid))
    print("launched pid:", proc.pid)
    monitor(proc, proc.pid)
    log_f.close()
    rc = proc.returncode if proc is not None else 1
    n_lines = sum(1 for _ in open(LOG, "r", errors="replace"))
    print("\n=== TRAINING EXITED rc=%s | log lines: %d ===" % (rc, n_lines))
    if rc != 0:
        print("TRAINING FAILED — last lines:")
        for l in open(LOG, "r", errors="replace").read().splitlines()[-12:]:
            print("   ", l[:180])
        raise SystemExit("Send the lines above to Hermes.")

# ── S8: convergence report ───────────────────────────────────────────
stage(8, "Convergence report")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
cks = sorted(glob.glob("/content/akan-vits-finetuned/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
state_file = None
if cks and Path(cks[-1], "trainer_state.json").exists():
    state_file = os.path.join(cks[-1], "trainer_state.json")
if not state_file:
    alt = sorted(glob.glob("/content/drive/MyDrive/**/trainer_state.json", recursive=True))
    if alt: state_file = alt[0]
if not state_file:
    print("convergence report skipped — no checkpoints yet")
else:
    st = json.load(open(state_file))
    hist = st.get("log_history", [])
    train = [(h["step"], h["loss"]) for h in hist if "loss" in h]
    evals = [(h["step"], v) for h in hist for k, v in h.items()
             if k.startswith("eval_") and isinstance(v, (int, float)) and not k.endswith("runtime")]
    print("steps trained:", st.get("global_step"), "| train pts:", len(train), "| eval pts:", len(evals))
    fig, ax = plt.subplots(figsize=(9, 4.5))
    if train: xs, ys = zip(*train); ax.plot(xs, ys, lw=1.5, label="train loss")
    if evals:
        k0 = "eval_loss" if any(k == "eval_loss" for _, k in evals) else evals[0][1]
        ax.plot([s for s, k in evals if k == k0], [v for s, k in evals if k == k0], "o-", label=k0)
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.grid(alpha=0.3); ax.legend()
    ax.set_title("Akan TTS fine-tune — convergence")
    fig.tight_layout()
    os.makedirs("/content/akan-vits-finetuned", exist_ok=True)
    fig.savefig("/content/akan-vits-finetuned/loss_curve.png", dpi=130)
    if len(train) >= 5:
        n5 = max(1, len(train)//10)
        drop = sum(y for _, y in train[:n5])/n5 - sum(y for _, y in train[-n5:])/n5
        tail = [y for _, y in train[-max(1, len(hist)//5):]]
        flat = (max(tail) - min(tail)) if tail else 0
        first10 = sum(y for _, y in train[:n5]) / n5
        verdict = ("CONVERGED - plateau reached" if drop <= 0.02*first10 and flat < 0.02*max(1e-9,last10 if False else first10)
                   else "CONVERGING - total drop %.4f" % drop)
        print("VERDICT:", verdict)
        json.dump({"global_step": st.get("global_step"), "verdict": verdict,
                   "state_file": state_file, "n_train_points": len(train)},
                  open("/content/akan-vits-finetuned/convergence_summary.json", "w"), indent=2)
    print("artifacts: loss_curve.png + convergence_summary.json")

# ── S9: save + push to Hub ───────────────────────────────────────────
stage(9, "Save to Drive + push artifacts to Hugging Face")
shell("zip -qr '/content/drive/MyDrive/UG_TTS/akan_tts_ckpt_new.zip' /content/akan-vits-finetuned || true", must=False)
from huggingface_hub import upload_file
for f in ["/content/akan-vits-finetuned/loss_curve.png",
          "/content/akan-vits-finetuned/convergence_summary.json",
          "/content/train_akan.log"]:
    if os.path.exists(f):
        try:
            upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f),
                        repo_id="Dickson32-cell/akan-twi-mms", repo_type="model")
            print("uploaded:", os.path.basename(f))
        except Exception as e:
            print("upload failed (keep the Drive copy):", f, str(e)[:120])
print("\n"
      "✅✅✅ DONE — model + training log are on the Hub at\n"
      "   https://huggingface.co/Dickson32-cell/akan-twi-mms\n"
      "Send the VERDICT line + 'TRAINING EXITED rc=' line to Hermes.")

In [ ]:
# 1. Install harness + build the Cython monotonic-align kernel (required for speed)
# CRITICAL: Colab preinstalls transformers 5.x but this harness requires 4.x
# (config attr handling). Pin FIRST, then clone + install.
!pip install -q 'transformers==4.49.0'
!git clone https://github.com/ylacombe/finetune-hf-vits
%cd finetune-hf-vits
!pip install -q -r requirements.txt
!pip install -q 'datasets[audio]>=3.2,<4.0' librosa soundfile accelerate
%cd monotonic_align
!mkdir -p monotonic_align && python setup.py build_ext --inplace
%cd ..
import transformers, datasets
print('VERSION CHECK -> transformers:', transformers.__version__, '| datasets:', datasets.__version__)
assert transformers.__version__.startswith('4.'), 'transformers 4.x required - re-run this cell'


In [ ]:
# 2. Authenticate with Hugging Face (write token) — needed to pull the discriminator and push the model
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3. Get the prepared corpus: manifest zip from Drive, then stream ONLY the
# selected rows' audio from the Hub (datacenter speed, ~10-20 min).
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/work && unzip -o -q '/content/drive/MyDrive/UG_TTS/tts_data.zip' -d /content/work
import urllib.request, pathlib
base = 'https://raw.githubusercontent.com/Dickson32-cell/akan-tts-asante/master/src/'
for f in ['normalize_twi.py', 'extract_asante_colab.py']:
    pathlib.Path('/content/' + f).write_text(
        urllib.request.urlopen(base + f, timeout=60).read().decode('utf-8'), encoding='utf-8')
print('scripts fetched from GitHub (single source of truth)')
!python /content/extract_asante_colab.py --manifest /content/work/selected.jsonl --out /content/tts_data


In [ ]:
# 4. Build the TRAINING checkpoint for 'aka': generator (hub) + Meta's original Akan discriminator
!python convert_original_discriminator_checkpoint.py --language_code aka --pytorch_dump_folder_path /content/aka_train_base
!ls /content/aka_train_base

In [ ]:
# 5. Training config (hyperparameters tuned for ~10 h single-speaker data on T4)
import json, pathlib
cfg = {
  "project_name": "akan_twi_tts",
  "push_to_hub": True,
  "hub_model_id": "Dickson32-cell/akan-twi-mms",   # fixed - no placeholder
  "report_to": ["tensorboard"],   # keep run data: tfevents land in output_dir
  "logging_strategy": "steps",
  "logging_steps": 20,             # loss line every 20 steps -> full curve survives
  "overwrite_output_dir": True,
  "output_dir": "/content/akan-vits-finetuned",

  "dataset_name": "/content/tts_data",
  "dataset_config_name": None,
  "audio_column_name": "audio",
  "text_column_name": "text",
  "train_split_name": "train",
  "eval_split_name": "eval",
  "speaker_id_column_name": "speaker_id",
  "filter_on_speaker_id": 0,
  "override_speaker_embeddings": False,

  "full_generation_sample_text": "Mema wo akye, me nuanom. Wo ho te sɛn?",

  "max_duration_in_seconds": 12.0,
  "min_duration_in_seconds": 2.0,
  "max_tokens_length": 500,

  "model_name_or_path": "/content/aka_train_base",

  "preprocessing_num_workers": 2,
  "do_train": True,
  "num_train_epochs": 30,
  "gradient_accumulation_steps": 1,
  "per_device_train_batch_size": 16,
  "learning_rate": 2e-4,
  "lr_scheduler_type": "cosine",
  "warmup_ratio": 0.02,
  "adam_beta1": 0.8,
  "adam_beta2": 0.99,
  "weight_decay": 0.01,
  "group_by_length": False,

  "do_eval": True,
  "eval_steps": 500,
  "per_device_eval_batch_size": 8,
  "max_eval_samples": 25,
  "do_step_schedule_per_epoch": False,
  "save_steps": 1000,
  "save_total_limit": 3,

  "weight_disc": 3,
  "weight_fmaps": 1,
  "weight_gen": 1,
  "weight_kl": 1.5,
  "weight_duration": 1,
  "weight_mel": 35,

  "fp16": True,
  "seed": 456
}
pathlib.Path('/content/finetune_akan.json').write_text(json.dumps(cfg, indent=2))
print('config written')

In [ ]:
# 6. TRAIN — just run this cell. It starts fresh or auto-continues if a run exists.
# Runs ~4-6 h on T4. Logs stream live; everything is saved to /content/train_akan.log.
import subprocess, time, os, glob

LOG = "/content/train_akan.log"
if not glob.glob("/content/akan-vits-finetuned/checkpoint-*"):
    print("No checkpoints found → starting a FRESH run.")
    base_cmd = ["python", "-m", "accelerate.commands.launch", "run_vits_finetuning.py",
                "/content/finetune_akan.json",
                "--save_steps", "1000", "--save_total_limit", "3",
                "--logging_steps", "20"]
else:
    print("Checkpoints found → CONTINUING from the last one.")
    base_cmd = ["python", "-m", "accelerate.commands.launch", "run_vits_finetuning.py",
                "/content/finetune_akan.json",
                "--save_steps", "1000", "--save_total_limit", "3",
                "--logging_steps", "20", "--resume_from_checkpoint", "true"]

log_f = open(LOG, "a")
proc = subprocess.Popen(base_cmd, stdout=log_f, stderr=subprocess.STDOUT,
                        bufsize=1, universal_newlines=True)
print("PID:", proc.pid, "| log:", LOG)

last_len = 0
while proc.poll() is None:
    time.sleep(20)
    try:
        with open(LOG, "r", errors="replace") as f:
            lines = f.read().splitlines()
        for l in lines[last_len:]:
            if any(k in l.lower() for k in ("loss", "eval_", "error", "traceback", "epoch", "it/s")):
                print(l[:200])
        last_len = len(lines)
    except FileNotFoundError:
        pass
log_f.close()
n_lines = sum(1 for _ in open(LOG, "r", errors="replace"))
print(f"\n=== TRAINING EXITED rc={proc.returncode} | log lines: {n_lines} ===")
if proc.returncode == 0:
    print("NOW: run the next cell (6b) for the convergence report, then the SAVE+PUSH cell.")
else:
    print("Training failed - copy the last lines above and send them to Hermes.")

In [ ]:
# 6b. CONVERGENCE REPORT — built from trainer_state.json (auto-written by Trainer)
# Evaluator asks: "did training converge?" — this cell answers it with curves + verdict.
import json, glob, os, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cks = sorted(glob.glob("/content/akan-vits-finetuned/checkpoint-*"),
             key=lambda p: int(p.rsplit("-", 1)[1]))
state_file = "/content/akan-vits-finetuned/checkpoint-final/trainer_state.json"
if cks and not os.path.exists(state_file):
    state_file = os.path.join(cks[-1], "trainer_state.json")   # latest checkpoint wins

if not os.path.exists(state_file):
    alt = sorted(glob.glob("/content/drive/MyDrive/**/trainer_state.json", recursive=True))
    if alt:
        state_file = alt[0]
        print("using Drive copy:", state_file)
    else:
        print("⛔⛔⛔  NOTHING IS WRONG — WRONG ORDER, THAT'S ALL  ⛔⛔⛔")
        print("⛔ This convergence cell only works AFTER the training cell has finished.")
        print("⛔⛔⛔  YOUR ACTION: scroll UP to the cell titled '6. TRAIN' and press its play ▶ button.  ⛔⛔⛔")
        print("If you already trained elsewhere: open cell 9 once (it mounts Drive), then re-run me \u2014 I will find trainer_state.json on Drive.")
        raise SystemExit(0)   # stop cleanly, no traceback
st = json.load(open(state_file))
hist = st.get("log_history", [])
train = [(h["step"], h["loss"]) for h in hist if "loss" in h]
evals = [(h["step"], v) for h in hist for k, v in h.items() if k.startswith("eval_") and isinstance(v, (int, float)) and not k.endswith("runtime")]

print(f"steps trained: {st.get('global_step')} | train-loss points: {len(train)} | eval points: {len(evals)}")

fig, ax = plt.subplots(figsize=(9, 4.5))
if train:
    xs, ys = zip(*train); ax.plot(xs, ys, label="train loss", lw=1.5)
if evals:
    k0 = "eval_loss" if any(k == "eval_loss" for _, k in evals) else evals[0][1]
    exs = [s for s, k in evals if k == k0]
    eys = [v for s, k in evals if k == k0]
    ax.plot(exs, eys, "o-", label=k0, lw=1.5)
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title("Akan TTS fine-tune \u2014 convergence")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
PNG = "/content/akan-vits-finetuned/loss_curve.png"
fig.savefig(PNG, dpi=130)

def verdict(train, evals):
    if len(train) < 5: return "INSUFFICIENT LOG POINTS"
    first10 = sum(y for _, y in train[:max(1, len(train)//10)]) / max(1, len(train)//10)
    last10  = sum(y for _, y in train[-max(1, len(train)//10):]) / max(1, len(train)//10)
    drop = first10 - last10
    tail = [y for _, y in train[-len(hist)//5 or 1:]]
    flat = max(tail) - min(tail) if tail else 0
    if drop <= 0 and flat > 0.2 * first10: return "NOT CONVERGING \u2014 loss rising/oscillating, check config & discriminator"
    if drop <= 0.02 * first10 and flat < 0.02 * last10: return "CONVERGED \u2014 plateau reached (further steps add little)"
    if drop > 0: return f"CONVERGING \u2014 total drop {drop:.4f} ({drop/first10*100:.1f}%); tail oscillation ±{flat:.4f}"
    return "CHECK CURVE MANUALLY"

v = verdict(train, evals)
tail_summary = ""
if train:
    tail_summary = f"last losses: {[round(y,4) for _, y in train[-5:]]}"
print("VERDICT:", v)
print(tail_summary)
json.dump({"global_step": st.get("global_step"), "verdict": v,
           "n_train_points": len(train), "n_eval_points": len(evals),
           "state_file": state_file},
          open("/content/akan-vits-finetuned/convergence_summary.json", "w"), indent=2)
print("saved: loss_curve.png + convergence_summary.json in /content/akan-vits-finetuned/")
print("Cell 9 uploads both to the Hub repo + Drive \u2014 evaluators see them beside the model.")


In [ ]:
# 7. Quick listening test — unseen Twi text (NOT from the corpus)
# NOTE: run ONLY after cell 6 finishes training (model is in /content/akan-vits-finetuned)
import sys, os, torch, soundfile as sf
if not os.path.isdir('/content/akan-vits-finetuned'):
    raise SystemExit('Training output not found yet - run cells 2,3,4,A,B (train) FIRST, then this cell.')
sys.path.insert(0, '/content')  # normalize_twi.py is in tts_data zip root
from normalize_twi import normalize_twi
from transformers import VitsModel, VitsTokenizer
M = '/content/akan-vits-finetuned'
tok = VitsTokenizer.from_pretrained(M)
model = VitsModel.from_pretrained(M).eval()
text = 'Mema wo akye, me nuanom. Wo ho te sɛn?'
inputs = tok(text=normalize_twi(text), return_tensors='pt')
with torch.no_grad():
    wav = model(**inputs).waveform[0]
sf.write('/content/demo_unseen.wav', (wav.numpy()*32767).astype('int16'), model.config.sampling_rate)
import IPython; IPython.display.Audio('/content/demo_unseen.wav')

In [ ]:
# 7b. ASR judge benchmark (round-trip evaluation judge selection)
# Whisper has no 'tw' language token -> use Akan/Twi fine-tuned checkpoints + English carrier prompt.
# Ground-truth clips: upload 06_samples/probe/*.wav to /content/probe/ with the truth dict below.
CANDIDATES = [
    'sadicko/whisper-small-akan-finetuned',
    'worldboss/whisper-small-twi-v1',
    'fiifinketia/whisper-large-v3-turbo-akan',
]
TRUTH = {
 'probe_0': '1 BERƐSOSƐM 1.',
 'probe_1': 'Ɛfiri Adam so de kɔsi Abraham so.',
 'probe_2': 'Adam, Set, Enos, Kenan, Mahalalel, Yared, Hanok, Metusela, Lamek, Noa, Sem, Ham ne Yafet.',
 'probe_3': 'Yafet mma ne: Gomer, Magog, Madai, Yawan, Tubal, Mesek ne Tiras.',
 'probe2_0': 'Ntonto a ɛtɔ so ɛnan no bɔɔ Seorim.',
 'probe2_1': 'Ntonto a ɛtɔ so enum no bɔɔ Malkia.',
 'probe2_2': 'Ntonto a ɛtɔ so nsia no bɔɔ Miyamin.',
}
import sys
sys.path.insert(0, '/content')
from normalize_twi import normalize_twi
from transformers import pipeline
import glob

def cer_(ref, hyp):
    r = list(normalize_twi(ref).replace(' ', '')); h = list(normalize_twi(hyp).replace(' ', ''))
    import editdistance
    return editdistance.eval(r, h) / max(1, len(r))

!pip install -q editdistance
results = {}
for cand in CANDIDATES:
    try:
        pipe = pipeline('automatic-speech-recognition', model=cand, device=0)
        cers = []
        for p in sorted(glob.glob('/content/probe/*.wav')):
            stem = p.split('/')[-1].replace('.wav','')
            hyp = pipe(p)['text'].strip()
            c = cer_(TRUTH.get(stem, ''), hyp)
            cers.append(c)
            print(cand, stem, round(c,2), '::', hyp[:60])
        results[cand] = sum(cers)/len(cers)
        del pipe
        import gc, torch; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(cand, 'FAILED:', e)
print('\nRANKING (mean CER):'); [print(f'  {m}: {c}') for m, c in sorted(results.items(), key=lambda kv: kv[1])]


In [ ]:
# 8. Save checkpoint + samples to Drive (survives session loss; also push_to_hub from cfg)
!zip -qr /content/drive/MyDrive/UG_TTS/akan_tts_ckpt_$(date +%m%d).zip /content/akan-vits-finetuned
print('saved to Drive')
# --- convergence artifacts to Hub + Drive (evaluator evidence) ---
from huggingface_hub import upload_file
for f in ["/content/akan-vits-finetuned/loss_curve.png",
          "/content/akan-vits-finetuned/convergence_summary.json",
          "/content/train_akan.log"]:
    if os.path.exists(f):
        try:
            upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f),
                        repo_id="Dickson32-cell/akan-twi-mms",
                        repo_type="model")
            print("uploaded:", os.path.basename(f))
        except Exception as e:
            print("upload failed (push manually from Drive copy):", f, e)
